# 01 - Data Collection

This notebook downloads adjusted price data for the oil benchmark, selected airlines, and the S&P 500 market benchmark. The raw adjusted close price dataset is saved to `data/raw/raw_prices.csv`.

## Imports and path setup

The notebook uses the shared project configuration in `src/config.py` for tickers, start date, and airline classifications.

In [1]:
from pathlib import Path
import sys

import pandas as pd
import yfinance as yf

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == 'notebooks':
    PROJECT_ROOT = PROJECT_ROOT.parent

SRC_DIR = PROJECT_ROOT / 'src'
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

RAW_DATA_DIR = PROJECT_ROOT / 'data' / 'raw'
RAW_DATA_DIR.mkdir(parents=True, exist_ok=True)
RAW_PRICES_PATH = RAW_DATA_DIR / 'raw_prices.csv'

print(f'Project root: {PROJECT_ROOT}')
print(f'Raw output path: {RAW_PRICES_PATH}')

Project root: /home/marcusai/MyProjects/Oil vs Airline Stocks Data Analysis
Raw output path: /home/marcusai/MyProjects/Oil vs Airline Stocks Data Analysis/data/raw/raw_prices.csv


## Load project configuration

The asset universe is defined once in `src/config.py` so later notebooks can use the same names and tickers.

In [2]:
from config import AIRLINE_CLASSIFICATION, START_DATE, TICKERS

expected_assets = list(TICKERS.keys())
ticker_symbols = list(TICKERS.values())
ticker_to_asset = {ticker: asset for asset, ticker in TICKERS.items()}

print('Start date:', START_DATE)
print('Assets and tickers:')
for asset, ticker in TICKERS.items():
    print(f'- {asset}: {ticker}')

print('\nAirline classifications loaded:')
for airline, classification in AIRLINE_CLASSIFICATION.items():
    print(f'- {airline}: {classification}')

Start date: 2018-01-01
Assets and tickers:
- Brent Oil: BZ=F
- Ryanair: RYAAY
- Lufthansa: LHA.DE
- Southwest: LUV
- American Airlines: AAL
- S&P 500: ^GSPC

Airline classifications loaded:
- Ryanair: {'region': 'Europe', 'business_model': 'Low-cost', 'hedging_profile': 'Historically higher hedging'}
- Lufthansa: {'region': 'Europe', 'business_model': 'Legacy', 'hedging_profile': 'Mixed / moderate hedging'}
- Southwest: {'region': 'United States', 'business_model': 'Low-cost', 'hedging_profile': 'Historically strong hedging'}
- American Airlines: {'region': 'United States', 'business_model': 'Legacy', 'hedging_profile': 'Historically lower / limited hedging'}


## Download adjusted price data

Yahoo Finance data is downloaded with `auto_adjust=False`, then the `Adj Close` field is selected. This keeps the raw dataset focused on adjusted close prices while preserving a clear data-source path.

In [3]:
downloaded = yf.download(
    tickers=ticker_symbols,
    start=START_DATE,
    auto_adjust=False,
    progress=False,
    threads=True,
)

if downloaded.empty:
    raise ValueError('Yahoo Finance returned an empty dataset.')

if isinstance(downloaded.columns, pd.MultiIndex):
    if 'Adj Close' in downloaded.columns.get_level_values(0):
        raw_prices = downloaded['Adj Close'].copy()
    else:
        raise KeyError('Downloaded data does not contain an Adj Close field.')
else:
    if 'Adj Close' in downloaded.columns:
        raw_prices = downloaded[['Adj Close']].copy()
        raw_prices.columns = ticker_symbols
    else:
        raise KeyError('Downloaded data does not contain an Adj Close field.')

raw_prices = raw_prices.rename(columns=ticker_to_asset)
raw_prices = raw_prices.reindex(columns=expected_assets)
raw_prices.index = pd.to_datetime(raw_prices.index)
raw_prices.index.name = 'Date'

print('Downloaded adjusted close prices.')
print('Shape:', raw_prices.shape)

Downloaded adjusted close prices.
Shape: (2181, 6)


## Basic inspection

Inspect the first rows, column names, data types, and overall structure before saving.

In [4]:
display(raw_prices.head())
print('\nDataFrame info:')
raw_prices.info()
print('\nColumns:', list(raw_prices.columns))

Ticker,Brent Oil,Ryanair,Lufthansa,Southwest,American Airlines,S&P 500
Date,,,,,,
2018-01-02,66.570000,40.315563,17.618053,59.443661,51.647560,2695.810059
2018-01-03,67.839996,41.089314,17.978792,58.206196,51.014027,2713.060059
2018-01-04,68.070000,41.322971,17.874062,58.017868,51.335659,2723.989990
2018-01-05,67.620003,41.614090,17.606419,57.677120,51.316170,2743.149902
2018-01-08,67.779999,41.951168,17.780970,57.390167,50.809349,2747.709961



DataFrame info:
<class 'pandas.DataFrame'>
DatetimeIndex: 2181 entries, 2018-01-02 to 2026-06-12
Data columns (total 6 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   Brent Oil          2126 non-null   float64
 1   Ryanair            2123 non-null   float64
 2   Lufthansa          2144 non-null   float64
 3   Southwest          2123 non-null   float64
 4   American Airlines  2123 non-null   float64
 5   S&P 500            2123 non-null   float64
dtypes: float64(6)
memory usage: 119.3 KB

Columns: ['Brent Oil', 'Ryanair', 'Lufthansa', 'Southwest', 'American Airlines', 'S&P 500']


## Missing-value check

Missing values are reported, not filled. Cleaning decisions belong in the next notebook.

In [5]:
missing_values = raw_prices.isna().sum().sort_values(ascending=False)
missing_percent = (raw_prices.isna().mean() * 100).sort_values(ascending=False).round(2)
missing_summary = pd.DataFrame({
    'missing_values': missing_values,
    'missing_percent': missing_percent,
})
display(missing_summary)

,missing_values,missing_percent
Ticker,,
Ryanair,58,2.66
Southwest,58,2.66
S&P 500,58,2.66
American Airlines,58,2.66
Brent Oil,55,2.52
Lufthansa,37,1.70


## Date-range check

Confirm the actual available date range from Yahoo Finance.

In [6]:
date_range = {
    'start': raw_prices.index.min(),
    'end': raw_prices.index.max(),
    'row_count': len(raw_prices),
}
date_range

{'start': Timestamp('2018-01-02 00:00:00'),
 'end': Timestamp('2026-06-12 00:00:00'),
 'row_count': 2181}

## Save raw data

Save the raw adjusted close price dataset. No returns are calculated in this phase.

In [7]:
raw_prices.to_csv(RAW_PRICES_PATH)
print(f'Saved raw prices to: {RAW_PRICES_PATH}')

Saved raw prices to: /home/marcusai/MyProjects/Oil vs Airline Stocks Data Analysis/data/raw/raw_prices.csv


## Phase 2 validation

Validate that the CSV exists, includes every expected asset, has a date index, and is not empty.

In [8]:
saved_prices = pd.read_csv(RAW_PRICES_PATH, parse_dates=['Date'], index_col='Date')

assert RAW_PRICES_PATH.exists(), 'raw_prices.csv was not created.'
assert not saved_prices.empty, 'raw_prices.csv is empty.'
assert isinstance(saved_prices.index, pd.DatetimeIndex), 'Date index was not preserved as dates.'
assert saved_prices.index.name == 'Date', 'Date index name is missing or incorrect.'
assert list(saved_prices.columns) == expected_assets, 'Saved assets do not match expected assets.'

validation_summary = {
    'csv_exists': RAW_PRICES_PATH.exists(),
    'shape': saved_prices.shape,
    'date_index_name': saved_prices.index.name,
    'date_range_start': saved_prices.index.min().strftime('%Y-%m-%d'),
    'date_range_end': saved_prices.index.max().strftime('%Y-%m-%d'),
    'assets': list(saved_prices.columns),
    'missing_values': saved_prices.isna().sum().to_dict(),
}
validation_summary

{'csv_exists': True,
 'shape': (2181, 6),
 'date_index_name': 'Date',
 'date_range_start': '2018-01-02',
 'date_range_end': '2026-06-12',
 'assets': ['Brent Oil',
  'Ryanair',
  'Lufthansa',
  'Southwest',
  'American Airlines',
  'S&P 500'],
 'missing_values': {'Brent Oil': 55,
  'Ryanair': 58,
  'Lufthansa': 37,
  'Southwest': 58,
  'American Airlines': 58,
  'S&P 500': 58}}